# Editing a sequence graph at forks and junctions

This notebook edits pUC19 to show what an edit can point at, how `insert`, `replace` and `delete`
use those targets, and how to control where an insertion goes when it lands on a junction.

## What an edit can point at

Substitutions and deletions are directed towards a sequence defined through:

- a `Locus`, from `graph.search(...)`, `annotation.locus` or an earlier edit,
- an `Annotation`, from `graph.annotations`,
- a region string, such as `"pUC19:100-110"`, `"MCS:1-4"` or `"pUC19:100.."`.

`replace(target, sequence)` swaps the target's sequence for `sequence`, and `delete(target)` removes it.

Insertions are targeted to a specific point, defined as a `Position` (`locus.start()`, `locus.end()`
or `locus[index]`) or a `SuperPosition`, which holds several points at once and is written
`position_a | position_b`. `insert(sequence, after=point)` puts the sequence right after the point
and `insert(sequence, before=point)` right before it. You can pass both `before` and `after` with SuperPositions to create combinatorial designs.

Adding or subtracting a number from a position gives a position relative to it, found by walking the
graph: `position + 3` is three points further along its strand. If a walk traverses a fork, the result will be a `SuperPosition` representing both posibilities.

Every call also takes two optional arguments. `message=` sets the text recorded for the edit in the
repository history. `stack=True` keeps the sequence being edited as an alternative route instead of
retiring it, which is introduced below. Each successful call records one operation, and `insert()`
and `replace()` return the `Locus` of the sequence they added, which is useful to annotate your edit.

In [ ]:
from pathlib import Path
import tempfile

import gen

repo = gen.Repository(tempfile.mkdtemp() + "/repository")
graph = repo.import_genbank(str(Path("puc19.gbk").resolve()))[0]

mcs = next(annotation for annotation in graph.annotations if annotation.name == "MCS")
locus = mcs.locus

## Replacing, history and stacking

The multiple cloning site (MCS) is 57 bases long. `locus[10:20]` is the ten bases at reading
positions 10 to 19. Replacing them retires the route through those bases and connects their
neighbours through the new sequence.

To see the edit, every plot below names the `Locus` that the edit returned. Wrapping it in an
`Annotation` and passing that to `fig.show()` highlights and labels it.

In [ ]:
replacement = graph.replace(locus[10:20], "GAGCTC")

replacement_annotation = gen.Annotation(replacement, "replacement")
fig = graph.plot(detail="full")
fig.show(replacement_annotation, "yellow")
fig

`plot()` draws the active graph, so the original ten bases are gone. Edits retire the connections they
replace rather than erasing them, and `show_history=True` brings them back, dimmed.

In [ ]:
fig = graph.plot(detail="full", show_history=True)
fig.show(replacement_annotation, "yellow")
fig

A plain edit retires the original route. With `stack=True`, an edit keeps the original route and adds
the new one beside it, so the graph holds both as alternatives. For `replace` and `insert` the added
route passes through the new sequence; for `delete` it skips the target. The current path, which
`export_fasta()` follows, stays as it was.

Stacking a replacement therefore creates a fork: the graph splits before the target and rejoins
after it. The original bases stay a valid target for later edits.

In [ ]:
alternative = graph.replace(locus[30:40], "TTAATTAA", stack=True)

alternative_annotation = gen.Annotation(alternative, "alternative")
fig = graph.plot(detail="full")
fig.show(alternative_annotation, "yellow")
fig

`delete(target)` works like `replace` without the new sequence and returns nothing, since there is
no new sequence to point at.

## Positions

A `Position` is a point in a node's sequence: `node` (the node slice it was read from), `offset`
within that slice, and `strand`. It is the everyday insert target: `locus.start()` and `locus.end()`
give the first and last position of a locus in reading order, on the locus's strand. Two positions
are equal when they name the same point of the same node, even after an edit has split the node.

`position_a | position_b` holds several positions together as a `SuperPosition`, and `insert()`
accepts either kind for `after` and `before`. A position knows the sequence graph its locus came
from, so `+` and `-` walk that graph. A step that reaches a fork gives a `SuperPosition` with one
position per branch, and a superposition holding several positions does not step again. Use
`position.on(other_graph)` to apply a position to a different graph, such as a copy of the sample.

The base just before the fork above is `locus[29:30]`. One step past it lands on both legs of the fork:

In [ ]:
trunk = locus[29:30]

print("last position before the fork:", trunk.end())
print("one step further:", trunk.end() + 1)

## Inserting on a single route

`insert(sequence, after=position)` connects the position to the new sequence and the new sequence to
whatever followed the position. `before=` mirrors it. Given both, `after` must read directly into
`before`, and the sequence goes exactly there. `insert` never removes sequence, so anchors further
apart are refused: use `replace` to swap out a stretch.

Here `TT` goes after the fifth base past the point where the fork rejoins, inside a single node.

In [ ]:
inserted = graph.insert("TT", after=locus[44:45].end())

inserted_annotation = gen.Annotation(inserted, "TT")
fig = graph.plot(detail="full")
fig.show(inserted_annotation, "yellow")
fig

## Inserting at a junction

A fork is a place where one block leads to several. The anchors of `insert` decide which of those
connections the new sequence goes on.

**Before the fork.** `after` names the last position before the fork. If several blocks follow it,
`after` alone attaches the new sequence to all of them, so one node sits in front of every leg. The
call below puts `GGATCC` on the shared stretch, ahead of both the original bases and the alternative.

In [ ]:
before_fork = graph.insert("GGATCC", after=trunk.end())

before_fork_annotation = gen.Annotation(before_fork, "before the fork")
fig = graph.plot(detail="full")
fig.show(before_fork_annotation, "yellow")
fig

**After the fork, on one leg.** Adding `before=` names the leg. `after` the new node and `before` the
first position of the alternative flank only the connection into the alternative, so the original
leg keeps its direct connection. `before=alternative.start()` alone would do the same, since stepping
back from the first position of the alternative reaches only what precedes it.

In [ ]:
on_one_leg = graph.insert("AAAA", after=before_fork.end(), before=alternative.start())

on_one_leg_annotation = gen.Annotation(on_one_leg, "one leg only")
fig = graph.plot(detail="full")
fig.show(on_one_leg_annotation, "yellow")
fig

A join works the same way in mirror image: `before` alone, given the first position after the join,
attaches the new sequence to the end of every arm, and adding `after=` the end of one arm keeps it on
that arm.

## Parallel chains: one insertion or one per chain

Two chains run side by side in one graph, `a` into `b` and `c` into `d`, and nothing connects one chain
to the other. The aim is to end up with `a`, `ATAT`, `b` and `c`, `ATAT`, `d`. The GFA below describes
the two chains, and is imported twice so that both ways of inserting can be compared.

In [ ]:
gfa = Path(tempfile.mkdtemp()) / "chains.gfa"
gfa.write_text(
    "H\tVN:Z:1.0\n"
    "S\ta\tAAAAA\nS\tb\tCCCCC\nS\tc\tGGGGG\nS\td\tTTTTT\n"
    "L\ta\t+\tb\t+\t0M\nL\tc\t+\td\t+\t0M\n"
    "P\tchain1\ta+,b+\t*\nP\tchain2\tc+,d+\t*\n"
)

shared = repo.import_gfa(str(gfa), sample="shared")
[a] = shared.search("AAAAA", sequence_kind="exact")
[c] = shared.search("GGGGG", sequence_kind="exact")

shared.plot(detail="full")

One `insert` adds one node, and that node is connected to everything its anchors name. A single
`after=a.end() | c.end()` therefore wires one `ATAT` node from both `a` and `c` into both `b` and `d`.
That also creates the crossover routes `a ATAT d` and `c ATAT b`, which neither chain had.

In [ ]:
shared_insert = shared.insert("ATAT", after=a.end() | c.end())

fig = shared.plot(detail="full")
fig.show(gen.Annotation(shared_insert, "one shared insert"), "yellow")
fig

To avoid the crossover, insert in a loop: one `insert` call per chain gives each chain its own connection
and nothing more. The graph ends up with two `ATAT` nodes holding the same sequence, one wired into each
chain, and no route from one chain into the other.

In [ ]:
separate = repo.import_gfa(str(gfa), sample="separate")

separate_inserts = []
for chain_start in ("AAAAA", "GGGGG"):
    [chain_locus] = separate.search(chain_start, sequence_kind="exact")
    separate_inserts.append(separate.insert("ATAT", after=chain_locus.end()))

fig = separate.plot(detail="full")
for number, separate_insert in enumerate(separate_inserts, start=1):
    fig.show(gen.Annotation(separate_insert, f"insert {number}"), ["yellow", "cyan"][number - 1])
fig